# BirdNET v2.4 — Post-Training Quantization (PTQ)

## Attempts and findings

**Dynamic range** (`birdnet_v2.4_int8.tflite`): weights INT8, activations float32. 14.2 MB, 72.5% reduction. Predictions wrong.

**Calibrated full INT8** (`birdnet_v2.4_int8_calibrated.tflite`): weights + activations INT8, 100-clip calibration. 14.1 MB, 72.8% reduction. 0/500 top-1 agreement.

**16x8** (`birdnet_v2.4_int8x16.tflite`): weights INT8, activations INT16, 100-clip calibration. 14.4 MB, 72.2% reduction. Predictions wrong.

**Why all INT8-weight approaches fail:** BirdNET's mel filterbank matrix contains many small values that INT8 (256 levels, scale ~0.008) rounds to zero, corrupting the spectrogram. The official Zenodo INT8 used quantization-aware training (QAT) — tensor names show `FakeQuantWithMinMaxVars` artifacts — not reproducible from the public SavedModel.

## Current: FP16 PTQ

`birdnet_v2.4_fp16.tflite` — weights stored as float16, used as float32 at runtime. No calibration needed. Float16 preserves the mel filterbank values accurately. Expected ~26 MB, ~50% reduction.

In [15]:
import os
import sys
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

sys.path.insert(0, os.path.dirname(os.getcwd()))
import config

MODEL_PATH = "/Users/qian/Library/Application Support/birdnet/acoustic-models/v2.4/pb/model-fp32"
OUTPUT_PATH = os.path.join(config.OUTPUTS_DIR, "models", "birdnet_v2.4_fp16.tflite")
SAMPLE_RATE = 48000
N_SAMPLES = 144000   # 3s × 48kHz — matches SavedModel input shape (None, 144000)
N_CALIB = 100
RANDOM_SEED = 42

print("TF version:", tf.__version__)

TF version: 2.20.0


In [11]:
labels = pd.read_csv(config.LABELS_PROGRESS_PATH)
species_clips = labels[labels["meaningful_source"] == "birdnet_species"].copy()

n_recorders = species_clips["Recorder"].nunique()
per_recorder = N_CALIB // n_recorders

calib_clips = (
    species_clips
    .groupby("Recorder", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), per_recorder), random_state=RANDOM_SEED))
    .sample(frac=1, random_state=RANDOM_SEED)
    .head(N_CALIB)
    .reset_index(drop=True)
)

print(f"Calibration clips: {len(calib_clips)}")
print(calib_clips["Recorder"].value_counts().to_string())

Calibration clips: 96
Recorder
Audio_Moth_6    16
Audio_Moth_5    16
Audio_Moth_3    16
Audio_Moth_1    16
Audio_Moth_2    16
Audio_Moth_4    16


/var/folders/b1/ntjf_3gn6zv2k1nf6cml66yw0000gn/T/ipykernel_83817/3506121571.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), per_recorder), random_state=RANDOM_SEED))


In [12]:
def load_audio(path):
    audio, sr = sf.read(path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if len(audio) < N_SAMPLES:
        audio = np.pad(audio, (0, N_SAMPLES - len(audio)))
    else:
        audio = audio[:N_SAMPLES]
    return audio.reshape(1, N_SAMPLES).astype(np.float32)


def representative_dataset():
    for _, row in calib_clips.iterrows():
        yield {"inputs": load_audio(row["audio_path"])}

In [13]:
import subprocess

SCRIPT_PATH = os.path.join(os.path.dirname(os.getcwd()), "scripts", "convert_birdnet_ptq.py")

print(f"Running: {SCRIPT_PATH}")
result = subprocess.run(
    [sys.executable, SCRIPT_PATH, OUTPUT_PATH],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:")
    print(result.stderr[-3000:])

Running: /Users/qian/KWF/rainforest-audio-detection/scripts/convert_birdnet_ptq.py


KeyboardInterrupt: 

In [16]:
TEST_CLIP = "/Users/qian/KWF/Segmented_Foldered/Audio Moth 1/0_4999/Audio_Moth_1_20250317_093627.wav"

# Load labels (6522 species from the SavedModel)
from birdnet.acoustic_models.v2_4.pb import AcousticPBDownloaderV2_4
_, species_list = AcousticPBDownloaderV2_4.get_model_path_and_labels("en_us")
species_labels = list(species_list)

interpreter = tf.lite.Interpreter(model_path=OUTPUT_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

audio = load_audio(TEST_CLIP)
interpreter.set_tensor(inp["index"], audio)
interpreter.invoke()
scores = interpreter.get_tensor(out["index"])[0]

top5_idx = np.argsort(scores)[::-1][:5]
print("Compressed model top-5 predictions:")
for i in top5_idx:
    print(f"  {species_labels[i]:<60} {scores[i]:.4f}")

print("\nFP32 baseline top-1: Hylophylax naevioides_Spotted Antbird  0.3415")
print(f"Compressed top-1 match: {species_labels[top5_idx[0]]}")

Compressed model top-5 predictions:
  Hylophylax naevioides_Spotted Antbird                        -0.6537
  Epinecrophylla fulviventris_Checker-throated Stipplethroat   -2.0604
  Oncostoma olivaceum_Southern Bentbill                        -2.6338
  Hylopezus perspicillatus_Streak-chested Antpitta             -2.6398
  Leptotila cassinii_Gray-chested Dove                         -2.7127

FP32 baseline top-1: Hylophylax naevioides_Spotted Antbird  0.3415
Compressed top-1 match: Hylophylax naevioides_Spotted Antbird


In [17]:
import time

FP32_TFLITE_PATH = "/Users/qian/Library/Application Support/birdnet/acoustic-models/v2.4/tf/model-fp32.tflite"
EVAL_N = 500

# Sample 500 clips from birdnet_species (fresh load, independent of b1000003)
labels_eval = pd.read_csv(config.LABELS_PROGRESS_PATH)
eval_clips = (
    labels_eval[labels_eval["meaningful_source"] == "birdnet_species"]
    .sample(EVAL_N, random_state=77)
    .reset_index(drop=True)
)
print(f"Evaluating {EVAL_N} clips...")

fp32_interp = tf.lite.Interpreter(model_path=FP32_TFLITE_PATH)
fp32_interp.allocate_tensors()
fp32_inp = fp32_interp.get_input_details()[0]
fp32_out = fp32_interp.get_output_details()[0]

int8_interp = tf.lite.Interpreter(model_path=OUTPUT_PATH)
int8_interp.allocate_tensors()
int8_inp = int8_interp.get_input_details()[0]
int8_out = int8_interp.get_output_details()[0]

def top1(interp, inp_d, out_d, audio):
    interp.set_tensor(inp_d["index"], audio)
    interp.invoke()
    return int(np.argmax(interp.get_tensor(out_d["index"])[0]))

agree = 0
t0 = time.time()
for i, (_, row) in enumerate(eval_clips.iterrows()):
    audio = load_audio(row["audio_path"])
    fp32_top1 = top1(fp32_interp, fp32_inp, fp32_out, audio)
    int8_top1 = top1(int8_interp, int8_inp, int8_out, audio)
    if fp32_top1 == int8_top1:
        agree += 1
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{EVAL_N}  agreement: {agree/(i+1)*100:.1f}%  ({time.time()-t0:.0f}s)")

print(f"\nTop-1 agreement: {agree}/{EVAL_N} = {agree/EVAL_N*100:.1f}%")

Evaluating 500 clips...


/Users/qian/miniforge3/envs/ds/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  100/500  agreement: 100.0%  (4s)
  200/500  agreement: 100.0%  (8s)
  300/500  agreement: 100.0%  (13s)
  400/500  agreement: 100.0%  (17s)
  500/500  agreement: 100.0%  (21s)

Top-1 agreement: 500/500 = 100.0%
